# Truy hồi ảnh bằng CLIP + GraphSAGE — demo từng bước

Notebook này dùng **cùng `RetrievalDemo` với giao diện web**, trên ảnh, vector và checkpoint thật. Bạn có thể thay câu ở bước 3 rồi chạy lại. Không huấn luyện hoặc chọn lại α/k ở đây.

Đầu vào: câu tiếng Anh + gallery ảnh và graph chú giải của từng ảnh. Đầu ra: top-k ảnh, điểm CLIP/graph/cuối và quan hệ có tác động khi bỏ riêng.

Cần Python 3.12 và `python -m pip install -r requirements-notebook.txt`. Chuẩn bị bằng `python -m scripts.run_demo check`, `images` và `prepare-model` theo README. Nếu dữ liệu ở checkout khác, sửa `ASSET_ROOT` trong ô dưới hoặc truyền `--asset-root` khi chạy notebook bằng CLI.

In [ ]:
import os
import sys
from pathlib import Path

assert sys.version_info >= (3, 12), "Notebook cần Python 3.12 trở lên."
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src/demo/service.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
ASSET_ROOT = Path(os.environ.get("GRAPH_DEMO_ASSET_ROOT", REPO_ROOT)).expanduser().resolve()
DEMO_CONFIG = Path(os.environ.get("GRAPH_DEMO_CONFIG", REPO_ROOT / "configs/demo/vg_coco.yaml"))
print("Code:", REPO_ROOT)
print("Dữ liệu/model:", ASSET_ROOT)

## 1. Kiểm tra dữ liệu và protocol

Các hash đầu vào phải khớp với cache và lượt train. Gallery mặc định là 1.500 ảnh test. **α và k đọc từ benchmark đã chọn trên validation**, không quét lại tham số bằng câu bạn đang thử.

In [ ]:
import json

from IPython.display import HTML, display

from src.demo.notebook import result_html
from src.demo.service import DemoConfig, RetrievalDemo, check_artifacts

options = DemoConfig.from_yaml(DEMO_CONFIG, asset_root=ASSET_ROOT)
protocol = check_artifacts(options)
assert protocol["selection"] == "validation"
assert protocol["available_images"] == protocol["gallery_images"]
print(json.dumps(protocol, ensure_ascii=False, indent=2))

## 2. Nạp một lần, dùng cho nhiều câu

CLIP đọc từ cache để chạy offline. GraphSAGE đọc đúng checkpoint của seed trong config. Vector ảnh và graph được sắp cùng image ID. Bước này không cập nhật trọng số.

In [ ]:
demo = RetrievalDemo.load(options)
print("Ảnh:", demo.image_vectors.shape)
print("Graph:", demo.graph_vectors.shape)
print("α =", demo.protocol["alpha"], "; k =", demo.protocol["candidate_k"])
assert demo.image_vectors.shape == demo.graph_vectors.shape

## 3. Nhập một câu mới và xem top-k

Thay `QUERY` bằng một mô tả tiếng Anh. Với câu tự nhập, chưa có nhãn đáp án nên notebook chỉ trình bày ảnh và điểm, không tự gọi kết quả là đúng/sai. Đây là truy hồi ảnh trong gallery, không phải sinh ảnh mới.

In [ ]:
QUERY = "A man riding a horse on a grassy field."
result = demo.search(QUERY, results=5, relations=3)
assert result["ground_truth"] is None
assert result["protocol"]["alpha"] == protocol["alpha"]
display(HTML(result_html(demo, result)))

## 4. Tách nhỏ công thức của cùng câu

1. CLIP text tạo vector query rồi chuẩn hóa L2: shape `(1, 512)`.
2. Lấy tích vô hướng với từng vector ảnh/graph đã chuẩn hóa → hai hàng điểm `(1, 1500)`.
3. Reranker chỉ trộn điểm trong top-k ứng viên CLIP. Ảnh ngoài nhóm giữ thứ tự CLIP ở phía sau. Query không được thêm làm node của graph.

Ô dưới dùng chính hàm rerank của thực nghiệm để kiểm thứ tự của kết quả demo.

In [ ]:
import numpy as np

from src.embeddings.store import l2_normalize
from src.retrieval.rerank import rerank_scores

q = l2_normalize(demo.encoder.encode_texts([QUERY]))
clip_scores = q @ demo.image_vectors.T
graph_scores = q @ demo.graph_vectors.T
final_scores = rerank_scores(clip_scores, graph_scores, protocol["alpha"], protocol["candidate_k"])
order = np.argsort(-final_scores[0], kind="stable")[:len(result["results"])]
assert [demo.image_ids[i] for i in order] == [r["image_id"] for r in result["results"]]
print("q:", q.shape, "; điểm CLIP:", clip_scores.shape, "; điểm graph:", graph_scores.shape)
row = result["results"][0]
mixed = protocol["alpha"] * row["clip_score"] + (1-protocol["alpha"]) * row["graph_score"]
assert row["rerank_candidate"] and np.isclose(mixed, row["final_score"])
print(f"Ảnh đầu: {protocol['alpha']} × {row['clip_score']:.5f} + "
      f"{1-protocol['alpha']:.2f} × {row['graph_score']:.5f} = {mixed:.5f}")

## 5. Quan hệ giải thích nói điều gì?

Với mỗi relation, giữ nguyên đối tượng nhưng bỏ relation đó rồi chạy lại GraphSAGE. `score_drop = điểm graph đầy đủ − điểm graph sau bỏ`.

- Số dương: relation hỗ trợ điểm của ảnh cho câu đang hỏi.
- Số âm: bỏ relation làm điểm tăng.
- Trong nhóm rerank, thay đổi điểm cuối bằng `(1 − α) × score_drop`; ảnh ngoài top-k CLIP không dùng graph cho điểm cuối.

Đây là tác động lên mô hình, không tự chứng minh relation đúng ngữ nghĩa. Mỗi trường hợp được đối chiếu lại bằng checkpoint đang dùng để tránh giải thích nhầm vector export.

In [ ]:
for relation in row["relations"]:
    print(f"{relation['subject']} → {relation['predicate']} → {relation['object']}")
    print(f"  Giảm điểm graph: {relation['score_drop']:+.6f}; "
          f"giảm điểm cuối: {relation['final_score_drop']:+.6f}")
    assert np.isclose(relation["final_score_drop"], (1-protocol["alpha"])*relation["score_drop"])

## 6. Ví dụ có đáp án: phân biệt hạng truy hồi với hạng sau can thiệp

Ví dụ được lấy theo thứ tự query ID cố định, không chọn theo kết quả tốt. Chỉ khi câu và ID khớp nguyên bản, demo mới dùng ground truth để báo hạng CLIP và hạng sau reranking. Các hạng này khác với việc theo dõi ảnh prediction sau xóa relation trong phép đo fidelity.

In [ ]:
example = demo.examples(limit=1)[0]
known = demo.search(example["text"], query_id=example["query_id"], results=3, relations=2)
assert known["ground_truth"] is not None
print(example["query_id"], known["ground_truth"])
display(HTML(result_html(demo, known)))

## 7. Lưu bằng chứng và mở giao diện

JSON lưu query, image ID, thứ hạng, ba loại điểm, relation và hash benchmark/checkpoint. Giữ cùng query/seed/model khi đưa hình và số liệu vào báo cáo.

Mở giao diện bằng `python -m scripts.run_demo serve` hoặc thêm `--asset-root /duong/dan/checkout-co-du-lieu`. Khi vấn đáp, thử một ví dụ có nhãn và một câu mới; đối chiếu CLIP với GraphSAGE rồi mở quan hệ giải thích.

In [ ]:
from src.data.json_io import write_json

output = REPO_ROOT / "runs/demo/notebook-query.json"
write_json(output, result)
print("Đã lưu:", output)
print("Notebook không huấn luyện, không chọn lại α/k và không sửa bảng thí nghiệm.")